# nb135 v8 — Boltz-2 cofolding (clean rewrite)

Install confirmed working via pip --target with numpy<2 pin. This version fixes the syntax error in the run loop that broke v7.

In [ ]:
import os, subprocess, sys, urllib.request, time
from pathlib import Path
os.environ['PYTHONUNBUFFERED'] = '1'

BATCH_IDX = int(os.environ.get('BATCH_IDX', '0'))
BATCH_SIZE = 1165
print(f'BATCH_IDX={BATCH_IDX}')

BZ_TARGET = '/kaggle/working/boltz_pkgs'
Path(BZ_TARGET).mkdir(exist_ok=True)
BOLTZ_BIN = f'{BZ_TARGET}/bin/boltz'

if not Path(BOLTZ_BIN).exists():
    print('Installing boltz to isolated target dir...')
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    '--target', BZ_TARGET, '-q',
                    'numpy==1.26.4', 'pandas==2.2.3', 'scipy==1.13.1',
                    'torch==2.4.0', 'torchmetrics==1.4.0', 'lightning==2.4.0', 'boltz', 'rdkit', 'pyyaml'], check=False)
    bin_dir = Path(f'{BZ_TARGET}/bin')
    if bin_dir.exists():
        for f in bin_dir.iterdir():
            f.chmod(0o755)

env = {**os.environ, 'PYTHONPATH': BZ_TARGET, 'PATH': f'{BZ_TARGET}/bin:' + os.environ.get('PATH','')}
r = subprocess.run([sys.executable, '-c',
    'import numpy, pandas; import boltz; print(f"numpy={numpy.__version__} pandas={pandas.__version__} boltz OK")'],
    env=env, capture_output=True, text=True, timeout=120)
print('Test stdout:', r.stdout.strip())
if r.returncode != 0:
    print('Test stderr:', r.stderr[-500:])

In [ ]:
PXR_SEQ = ('LDRRTVVPATQHVTGTAYIWYRSGLCEHHIVEAATRGNVMTPSCKLITEELLGRPVHIVQPVKAVCS'
           'IVKQSDCRPFNQRSFKKYFTMENKVMVLNQELIKLALNFKLQDGRPHGGIIYDLSGEEDPKSWIWE'
           'VLEAWDIKAQVGPVTYAVTSLPFLQLSQYLDQDLALYIHQAFRYGPNALLDLLTDTRKHADRLELN'
           'GLAIRLLPELEVALMLLTQHTLREEKAGNFETIAEPFNALVMQVMEGYREKDPEAKQNQELHIWAN'
           'KTKDPLLLEAHALDQFSCK')
import pandas as pd
HF = 'https://huggingface.co/datasets/openadmet/pxr-challenge-train-test/resolve/main'
for fn in ['pxr-challenge_TRAIN.csv', 'pxr-challenge_TEST_BLINDED.csv']:
    p = f'/kaggle/working/{fn}'
    if not Path(p).exists():
        urllib.request.urlretrieve(f'{HF}/{fn}', p)
tr = pd.read_csv('/kaggle/working/pxr-challenge_TRAIN.csv')
te = pd.read_csv('/kaggle/working/pxr-challenge_TEST_BLINDED.csv')
all_c = pd.concat([
    tr[['Molecule Name','SMILES']].assign(split='train'),
    te[['Molecule Name','SMILES']].assign(split='test'),
], ignore_index=True).rename(columns={'Molecule Name':'name','SMILES':'smiles'}).dropna()
lo = BATCH_IDX * BATCH_SIZE
hi = (BATCH_IDX + 1) * BATCH_SIZE
batch = all_c.iloc[lo:hi].reset_index(drop=True)
print(f'Batch: {len(batch)} compounds')

In [ ]:
import yaml, json
YAML_DIR = Path('/kaggle/working/boltz_in')
YAML_DIR.mkdir(exist_ok=True)
OUT_DIR = Path('/kaggle/working/boltz_out')
OUT_DIR.mkdir(exist_ok=True)

def safe(n):
    return ''.join(c if c.isalnum() else '_' for c in str(n))

for _, row in batch.iterrows():
    cfg = {
        'version': 1,
        'sequences': [
            {'protein': {'id': 'A', 'sequence': PXR_SEQ}},
            {'ligand': {'id': 'B', 'smiles': row['smiles']}},
        ],
        'properties': [{'affinity': {'binder': 'B'}}],
    }
    (YAML_DIR / f'{safe(row["name"])}.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))
print(f'Wrote {len(batch)} YAMLs')

In [ ]:
def find_aff(d):
    for jf in Path(d).glob('**/*affinity*.json'):
        try:
            j = json.load(open(jf))
            return j.get('affinity_pred_value') or j.get('affinity_pred') or j.get('affinity')
        except Exception:
            pass
    return None

# Diagnostic: run boltz on FIRST yaml, show full stderr if it fails
test_yaml = sorted(YAML_DIR.glob('*.yaml'))[0]
test_out = Path('/kaggle/working/boltz_test')
test_out.mkdir(exist_ok=True)
test_cmd = [BOLTZ_BIN, 'predict', str(test_yaml), '--out_dir', str(test_out),
            '--use_msa_server', '--diffusion_samples', '1',
            '--recycling_steps', '1', '--sampling_steps', '50']
print('TEST_CMD:', ' '.join(test_cmd))
test_r = subprocess.run(test_cmd, env=env, capture_output=True, text=True, timeout=900)
print('TEST_RC:', test_r.returncode)
print('TEST_STDOUT_tail:', test_r.stdout[-800:])
print('TEST_STDERR_tail:', test_r.stderr[-1500:])
print('TEST_AFF:', find_aff(test_out))

In [ ]:
# Only run full batch if test succeeded
if test_r.returncode != 0:
    print('Test failed; not running full batch')
else:
    print('Test OK; running full batch...')
    import shutil
    shutil.rmtree(test_out, ignore_errors=True)
    results = []
    t0 = time.time()
    for i, row in batch.iterrows():
        name = row['name']
        s = safe(name)
        out_p = OUT_DIR / s
        if out_p.exists():
            a = find_aff(out_p)
            if a is not None:
                results.append((name, a))
                continue
        cmd = [BOLTZ_BIN, 'predict', str(YAML_DIR / f'{s}.yaml'),
               '--out_dir', str(out_p), '--use_msa_server',
               '--diffusion_samples', '1', '--recycling_steps', '1', '--sampling_steps', '50']
        try:
            subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=900)
        except subprocess.TimeoutExpired:
            pass
        results.append((name, find_aff(out_p)))
        if (i + 1) % 5 == 0:
            elapsed = time.time() - t0
            eta_h = elapsed / (i + 1) * (len(batch) - i - 1) / 3600
            print(f'{i+1}/{len(batch)} elapsed={elapsed/60:.1f}m ETA={eta_h:.1f}h last={results[-1][1]}')
            pd.DataFrame(results, columns=['name', 'boltz_affinity']).to_parquet(
                f'/kaggle/working/boltz_batch{BATCH_IDX}_partial.parquet', index=False)
    pd.DataFrame(results, columns=['name', 'boltz_affinity']).to_parquet(
        f'/kaggle/working/boltz_batch{BATCH_IDX}.parquet', index=False)
    n_ok = sum(r[1] is not None for r in results)
    print(f'Done. {len(results)} results, non-NaN: {n_ok}')